<a href="https://colab.research.google.com/github/gnutan181/Training-model/blob/main/FinetuningofQwenwithlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q trl peft transformers datasets accelerate

from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,

    pipeline,

)
import torch

from trl import SFTConfig, SFTTrainer
import json
from peft import LoraConfig, TaskType, get_peft_model
from pathlib import Path
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 103.6 MB/s eta 0:00:00


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Load the dataset
# ─────────────────────────────────────────────────────────────────────────────
base_dir = Path.cwd()
data_path = base_dir / "qa360_sft.jsonl"

records = []
with data_path.open("r") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping malformed line due to error: {e}")

print(f"Total record loaded: {len(records)}")

Total record loaded: 4790


In [3]:

# ─────────────────────────────────────────────────────────────────────────────
# 1a. Format records into ChatML
# ─────────────────────────────────────────────────────────────────────────────

def format_record(record):
    instruction = record["instruction"]
    requirement = record["input"]
    analysis = json.dumps(record["output"], indent=2)

    return {
        "text": (
            f"<|im_start|>system\n{instruction}<|im_end|>\n"
            f"<|im_start|>user\n{requirement}<|im_end|>\n"
            f"<|im_start|>assistant\n{analysis}<|im_end|>"
        )
    }

formatted = [format_record(r) for r in records]

dataset = Dataset.from_list(formatted)



In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:


# # ─────────────────────────────────────────────────────────────────────────────
# # 2. Tokenization
# # ─────────────────────────────────────────────────────────────────────────────

model_checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [6]:
# # ─────────────────────────────────────────────────────────────────────────────
# #  3. Train / Test Split (90/10)
# # ─────────────────────────────────────────────────────────────────────────────

split= dataset.train_test_split(test_size=0.1, seed =42)
train_data = split["train"]
eval_data = split["test"]

In [7]:

# # ─────────────────────────────────────────────────────────────────────────────
# # 4.Building the Model
# # ─────────────────────────────────────────────────────────────────────────────

if torch.has_mps:
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model = AutoModelForCausalLM.from_pretrained(model_checkpoint,torch_dtype=torch.float16)


/tmp/ipykernel_269/2254790302.py:5: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  if torch.has_mps:
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
!pip install -U -q torchao

# ─────────────────────────────────────────────────────────────────────────────
# 5. LoRA Configuration
# ─────────────────────────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n================== BEFORE PEFT ===================")
print(f"Total Params: {total_params}")
print(f"Total Trainable Params: {trainable_params}")

peft_config = LoraConfig(
    task_type= TaskType.CAUSAL_LM,
    r=16,
    lora_alpha= 32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(model, peft_config)
print("\n================== AFTER PEFT ===================")
model.print_trainable_parameters()

model = model.to(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.5 MB/s eta 0:00:00



================== BEFORE PEFT ===================
Total Params: 1543714304
Total Trainable Params: 1543714304

================== AFTER PEFT ===================
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [9]:
# # ─────────────────────────────────────────────────────────────────────────────
# # 6.Training Args
# # ─────────────────────────────────────────────────────────────────────────────

batch_size = 4
training_dir = "qwen_qa360_lora_output"

# Force FP16 on older CUDA devices like T4 (Compute Capability 7.5)
use_fp16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

sft_config = SFTConfig(
    output_dir=training_dir,
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    disable_tqdm=False,
    gradient_accumulation_steps=4,
    bf16=use_bf16,
    fp16=use_fp16,
    dataset_text_field="text",
    max_length=2048,
    dataset_num_proc=1
)

In [10]:
# # ─────────────────────────────────────────────────────────────────────────────
# # 7. Trainer
# # ─────────────────────────────────────────────────────────────────────────────
# Cast model to float16 explicitly to match fp16 training configuration
model = model.to(torch.float16)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer
)

Adding EOS to train dataset (num_proc=1):   0%|          | 0/4311 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/4311 [00:00<?, ? examples/s]

Building labels for train dataset (num_proc=1):   0%|          | 0/4311 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=1):   0%|          | 0/4311 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset (num_proc=1):   0%|          | 0/4311 [00:00<?, ? examples/s…

Adding EOS to eval dataset (num_proc=1):   0%|          | 0/479 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=1):   0%|          | 0/479 [00:00<?, ? examples/s]

Building labels for eval dataset (num_proc=1):   0%|          | 0/479 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=1):   0%|          | 0/479 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset (num_proc=1):   0%|          | 0/479 [00:00<?, ? examples/s]

In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # 8. Training
# # ─────────────────────────────────────────────────────────────────────────────
import os
# Disable TRL optimized kernels that force custom BFloat16 precision on older GPUs
os.environ["TRL_USE_RICH"] = "0"

# Ensure trainable adapter parameters are in float32 while frozen weights are float16
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
    else:
        param.data = param.data.to(torch.float16)

# Adjust hyperparameters on the fly to fit into T4 GPU VRAM limits
trainer.args.per_device_train_batch_size = 1
trainer.args.gradient_accumulation_steps = 16  # Maintain effective batch size of 16 (1 * 16)

# Safely re-enable gradient checkpointing on the model to reduce activation memory usage
trainer.model.gradient_checkpointing_enable()
trainer.args.gradient_checkpointing = True
if hasattr(trainer.model, "config"):
    trainer.model.config.use_cache = False

trainer.args.bf16 = False
trainer.args.fp16 = True

# Fallback TRL's custom addmm behavior if it attempts to use out_dtype with BFloat16
import trl.kernels.chunked_logprob
trl.kernels.chunked_logprob._MM_OUT_DTYPE = False

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


In [ ]:

# # ─────────────────────────────────────────────────────────────────────────────
# # 9. Evaluate the model
# # ─────────────────────────────────────────────────────────────────────────────
result = trainer.evaluate()
print(result)


In [ ]:

# # ─────────────────────────────────────────────────────────────────────────────
# # 10. Save the Model
# # ─────────────────────────────────────────────────────────────────────────────
SAVE_DIR = "qwen_qa360_lora_model"
trainer.save_model(SAVE_DIR)


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# 11. Inferencing the locally saved Model
# ─────────────────────────────────────────────────────────────────────────────
generator = pipeline(
    'text-generation',
    model=SAVE_DIR,
    max_new_tokens=3080,
    device=device)


test_requirements = [
    "Admin can force password reset",
    "User I can upload a profile picture",
    "user can lock account after failed attempts"
]

instruction = records[0]["instruction"]

for requirement in test_requirements:
    prompt = (
        f"<|im_start|>system\n{instruction}<|im_end|>\n"
        f"<|im_start|>user\n{requirement}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    output = generator(
        prompt,
        do_sample = False,
        pad_token_id = tokenizer.eos_token_id
    )

    generated = output[0]["generated_text"]
    response = generated.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    print(f"\n Requirement: {requirement}")
    print(f"QA360 Analysis:\n{response}")
    print("-"*60)